# Refactored analysis notebook

This notebook is a clean entry point for the refactored Machine Learning coursework code.
It focuses on the three method modules that are most worth preserving:

1. **Clan classification** via Fisher-score screening + 2D KNN  
2. **Within-clan rank rule recovery** via single-feature threshold search  
3. **Prowess formula recovery** via sparse integer linear search  

The tournament simulation is retained only as an optional extension.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from interpreterule.config import DEFAULT_CLAN_ORDER, DEFAULT_FEATURES
from interpreterule.data import load_dataset, split_labeled_unlabeled
from interpreterule.clan_classifier import KNNClanClassifier
from interpreterule.rank_rules import ClanRankRuleModel
from interpreterule.prowess import IntegerLinearFormulaModel


In [ ]:
data = load_dataset(ROOT / "data" / "raw" / "machlearnia.csv")
split = split_labeled_unlabeled(data)
train_df = split.train
test_df = split.test

print(train_df.shape, test_df.shape)
train_df.head()

## 1. Clan classification

In [ ]:
clan_model = KNNClanClassifier()
clan_result = clan_model.fit(train_df)

print("Selected features:", clan_result.selected_features)
print("Best k:", clan_result.best_k)
print("Train accuracy:", round(clan_result.train_accuracy, 4))
clan_result.cv_summary

In [ ]:
clan_result.train_confusion

In [ ]:
clan_predictions, clan_confidence = clan_model.predict_with_confidence(test_df)
pd.DataFrame({
    "predicted_clan": clan_predictions,
    "vote_share_confidence": clan_confidence,
}).head()

## 2. Rank rule recovery

In [ ]:
rank_model = ClanRankRuleModel()
rank_model.fit(train_df)
rank_accuracy, rank_confusion = rank_model.training_report(train_df)

print("Train accuracy:", round(rank_accuracy, 4))
rank_model.rule_table().sort_values("clan").reset_index(drop=True)

In [ ]:
rank_confusion

In [ ]:
rank_predictions = rank_model.predict(test_df, clan_predictions)
pd.Series(rank_predictions).value_counts().sort_index()

## 3. Prowess formula recovery

In [ ]:
formula_model = IntegerLinearFormulaModel()
formula_result = formula_model.fit(train_df)

print(formula_result.formula_string)
print("CV MAE:", round(formula_result.cv_mae, 2))
print("Train RMSE:", round(formula_result.train_rmse, 2))
print("Train R^2:", round(formula_result.train_r2, 4))
formula_model.coefficient_table()

In [ ]:
prowess_predictions = formula_model.predict(test_df)
prowess_predictions[:10]

## 4. Reconstruct the full answer table

In [ ]:
answers = data.copy()
answers.loc[answers["clan"].isna(), "clan"] = clan_predictions
answers.loc[answers["rank"].isna(), "rank"] = rank_predictions
answers.loc[answers["prowess"].isna(), "prowess"] = prowess_predictions
answers.tail()

## 5. Saved outputs from the pipeline

In [ ]:
metrics_path = ROOT / "outputs" / "metrics.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

metrics["prowess_formula"]